# Data Extraction

This notebook extracts structured tabular data from the raw JSON files collected in `01_collection.ipynb`. Each raw file contains a GeoJSON response from the IEM Convective Outlook Warning (COW) API for a single NWS Weather Forecast Office and calendar year. The extraction process flattens those responses into two analysis-ready tables: `events`, containing one row per warning event, and `stormreports`, containing one row per Local Storm Report (LSR). Both tables are written as CSV files to `data/02_extraction/`, which serves as an immutable checkpoint of the source data before any cleaning or transformation is applied. No analytical decisions are made here; the output reflects the content of the source data as faithfully as possible.

## Implementation

Extraction is implemented by `COWExtractor` in `src/extraction/extractor.py`. It reads every `{WFO}_{YEAR}.json` file from `data/01_collection/COW/`, flattens the GeoJSON feature properties into rows, and writes two CSV files to `data/02_extraction/` as an immutable checkpoint before any cleaning decisions are applied.

| Method | Output |
|---|---|
| `extract_events()` | One row per warning event; `wfo` and `year` derived from filename |
| `extract_stormreports()` | One row per LSR; `warned=False` rows are unwarned storm reports (misses) |

Geometry is excluded from both tables because the analysis is statistical, not spatial. API-supplied `wfo` and `year` fields are replaced by filename-derived values to ensure consistency across all files.

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from extraction import COWExtractor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: raw JSON files written by 01_collection.ipynb
COW_DIR = Path("../data/01_collection/COW")

# Output: flattened CSV files, immutable checkpoint before cleaning.
# Cleaning decisions are applied in 03_cleaning.ipynb → data/03_cleaning/
EXTRACTED_DIR = Path("../data/02_extraction")

## Events

One row per warning event issued by a WFO. Key fields for analysis: `phenomena`, `issue`, `expire`, `verify`, and `lead0`. The `lead0` column is null for unverified events (i.e., those with no matching LSR).

In [2]:
extractor = COWExtractor(raw_dir=COW_DIR, extracted_dir=EXTRACTED_DIR)
print(extractor)

COWExtractor(raw=../data/01_collection/COW, extracted=../data/02_extraction)


In [3]:
events = extractor.extract_events()
print(f"{len(events):,} rows × {len(events.columns)} columns")
events.head()

10:00:05  INFO     Extracting events from ABQ_2020.json ...
10:00:05  INFO     Extracting events from ABQ_2021.json ...
10:00:05  INFO     Extracting events from ABQ_2022.json ...
10:00:05  INFO     Extracting events from ABQ_2023.json ...
10:00:05  INFO     Extracting events from ABQ_2024.json ...
10:00:05  INFO     Extracting events from ABQ_2025.json ...
10:00:05  INFO     Extracting events from ABR_2020.json ...
10:00:05  INFO     Extracting events from ABR_2021.json ...
10:00:05  INFO     Extracting events from ABR_2022.json ...
10:00:05  INFO     Extracting events from ABR_2023.json ...
10:00:05  INFO     Extracting events from ABR_2024.json ...
10:00:05  INFO     Extracting events from ABR_2025.json ...
10:00:05  INFO     Extracting events from AFC_2020.json ...
10:00:05  WARNING  AFC_2020.json: no events found, skipping
10:00:05  INFO     Extracting events from AFC_2021.json ...
10:00:05  WARNING  AFC_2021.json: no events found, skipping
10:00:05  INFO     Extracting events fro

161,481 rows × 32 columns


,wfo,year,phenomena,eventid,sharedborder,issue,expire,statuses,fcster,svr_tornado_possible,...,stormreports,stormreports_all,verify,tor_in_svrtorpossible,lead0,areaverify,visual_imgurl,product_text,product_href,link
0,ABQ,2020,TO,1,40531.482739,2020-03-13T23:04:00Z,2020-03-13T23:30:00Z,EXP,DPorter,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
1,ABQ,2020,TO,2,0.000000,2020-03-13T23:20:00Z,2020-03-13T23:32:00Z,CAN,DPorter,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
2,ABQ,2020,SV,1,54148.214946,2020-03-17T12:26:00Z,2020-03-17T12:38:00Z,CAN,33,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
3,ABQ,2020,FF,1,20057.998453,2020-04-26T20:52:00Z,2020-04-26T20:56:00Z,CAN,33,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...
4,ABQ,2020,SV,2,0.000000,2020-05-11T01:33:00Z,2020-05-11T02:15:00Z,CON,15,False,...,,,False,False,NaN,0.0,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2020-...


In [4]:
extractor.save(events, "events.csv")

10:00:12  INFO     Saved 161,481 rows to ../data/02_extraction/events.csv


PosixPath('../data/02_extraction/events.csv')

In [5]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 161481 entries, 0 to 161480
Data columns (total 32 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   wfo                    161481 non-null  str    
 1   year                   161481 non-null  int64  
 2   phenomena              161481 non-null  str    
 3   eventid                161481 non-null  int64  
 4   sharedborder           161479 non-null  float64
 5   issue                  161481 non-null  str    
 6   expire                 161481 non-null  str    
 7   statuses               161481 non-null  str    
 8   fcster                 158346 non-null  str    
 9   svr_tornado_possible   161481 non-null  bool   
 10  significance           161481 non-null  str    
 11  hailtag                136503 non-null  object 
 12  windtag                120458 non-null  object 
 13  carea                  161481 non-null  float64
 14  ar_ugc                 161481 non-null  object 

## Storm Reports

One row per Local Storm Report (LSR). The `warned` boolean indicates whether a warning was in effect at the time of the report. Warned reports link back to the events table via the `events` foreign key column; unwarned reports (`warned=False`) have null `events` and `leadtime` values and represent misses in POD calculations.

In [6]:
stormreports = extractor.extract_stormreports()
print(f"{len(stormreports):,} rows × {len(stormreports.columns)} columns")
stormreports.head()

10:00:12  INFO     Extracting stormreports from ABQ_2020.json ...
10:00:12  INFO     Extracting stormreports from ABQ_2021.json ...
10:00:12  INFO     Extracting stormreports from ABQ_2022.json ...
10:00:12  INFO     Extracting stormreports from ABQ_2023.json ...
10:00:12  INFO     Extracting stormreports from ABQ_2024.json ...
10:00:12  INFO     Extracting stormreports from ABQ_2025.json ...
10:00:12  INFO     Extracting stormreports from ABR_2020.json ...
10:00:12  INFO     Extracting stormreports from ABR_2021.json ...
10:00:12  INFO     Extracting stormreports from ABR_2022.json ...
10:00:12  INFO     Extracting stormreports from ABR_2023.json ...
10:00:12  INFO     Extracting stormreports from ABR_2024.json ...
10:00:12  INFO     Extracting stormreports from ABR_2025.json ...
10:00:12  INFO     Extracting stormreports from AFC_2020.json ...
10:00:12  WARNING  AFC_2020.json: no stormreports found, skipping
10:00:12  INFO     Extracting stormreports from AFC_2021.json ...
10:00:12  

236,800 rows × 19 columns


,wfo,year,valid,type,magnitude,city,county,state,source,remark,typetext,lon0,lat0,events,tdq,warned,leadtime,lsrtype,link
0,ABQ,2020,2020-03-17T23:24:00Z,T,NaN,PLEASANT HILL,CURRY,NM,SOCIAL MEDIA,NaN,TORNADO,-103.07,34.52,,False,False,NaN,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
1,ABQ,2020,2020-03-21T20:30:00Z,T,NaN,6 S KIRTLAND,SAN JUAN,NM,PUBLIC,LANDSPOUT TORNADO OVER NAVAJO AGRICULTURAL PRO...,TORNADO,-108.34,36.65,,False,False,NaN,TO,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
2,ABQ,2020,2020-05-16T23:53:00Z,H,1.0,3 SSW MAXWELL,COLFAX,NM,TRAINED SPOTTER,NaN,HAIL,-104.57,36.50,,False,False,NaN,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
3,ABQ,2020,2020-05-17T00:02:00Z,H,1.0,2 S MAXWELL,COLFAX,NM,SOCIAL MEDIA,NaN,HAIL,-104.54,36.51,2020ABQ10SVW1,False,True,3.0,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...
4,ABQ,2020,2020-05-17T02:14:00Z,H,1.25,8 WSW ROY,HARDING,NM,TRAINED SPOTTER,NaN,HAIL,-104.34,35.92,2020ABQ14SVW1,False,True,46.0,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/202...


In [7]:
extractor.save(stormreports, "stormreports.csv")

10:00:19  INFO     Saved 236,800 rows to ../data/02_extraction/stormreports.csv


PosixPath('../data/02_extraction/stormreports.csv')

In [8]:
stormreports.info()

<class 'pandas.DataFrame'>
RangeIndex: 236800 entries, 0 to 236799
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   wfo        236800 non-null  str    
 1   year       236800 non-null  int64  
 2   valid      236800 non-null  str    
 3   type       236800 non-null  str    
 4   magnitude  78384 non-null   object 
 5   city       236800 non-null  str    
 6   county     236800 non-null  str    
 7   state      236800 non-null  str    
 8   source     236800 non-null  str    
 9   remark     209678 non-null  object 
 10  typetext   236800 non-null  str    
 11  lon0       236800 non-null  float64
 12  lat0       236800 non-null  float64
 13  events     236800 non-null  str    
 14  tdq        236800 non-null  bool   
 15  warned     236800 non-null  bool   
 16  leadtime   181826 non-null  object 
 17  lsrtype    236800 non-null  str    
 18  link       236800 non-null  str    
dtypes: bool(2), float64(2), int64(1), 

## Storm Report Counts by Year and Type

In [9]:
counts = (
    stormreports
    .groupby(["year", "lsrtype"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"FF": "Flash Flood (FF)", "SV": "Severe Thunderstorm (SV)", "TO": "Tornado (TO)"})
)
counts["Total"] = counts.sum(axis=1)
counts.loc["Total"] = counts.sum()
counts

lsrtype,Flash Flood (FF),Severe Thunderstorm (SV),Tornado (TO),Total
year,,,,
2020,4487,31471,1574,37532
2021,6494,24950,1678,33122
2022,4247,28414,1682,34343
2023,4529,37893,1778,44200
2024,6552,34240,2262,43054
2025,7702,35113,1734,44549
Total,34011,192081,10708,236800
